In [15]:
import numpy as np
import random

#Isolation Tree
class Node:
  def __init__(self):
    self.size = None
    self.left = None
    self.right = None
    self.split_dim = None
    self.split_thresh = None
  def isolation_tree(self, X, h, h_max):
    X_left = []
    X_right = []
    if h >= h_max or len(X) <= 1:
      self.size = len(X)
    else:
      q = random.randint(1, len(X[0]))
      p = random.randint(int(X[:, q - 1].min()), int(X[:, q - 1].max()))
      for i in range(len(X)):
        if X[i][q - 1] >= p:
          X_left.append(X[i])
        else:
          X_right.append(X[i])
      X_left = np.array(X_left)
      X_right = np.array(X_right)
      self.left = Node()
      self.left.isolation_tree(X_left, h + 1, h_max)
      self.right = Node()
      self.right.isolation_tree(X_right, h + 1, h_max)
      self.split_dim = q
      self.split_thresh = p

def sample(X, sample_size):
    index = random.sample(range(len(X)), sample_size)
    return X[index]

#Isolation Forest
def isolation_forest(X, n, sample_size):
  forest = []
  h_max = int(np.log2(sample_size))
  for i in range(n):
    X_sample = sample(X, sample_size)
    root = Node()
    root.isolation_tree(X_sample, 0, h_max)
    forest.append(root)
  return forest

#Implementar

#Anomaly_Score
def anomaly_score(forest, x):
    lengths = [path_length(x, tree, 0) for tree in forest]
    return np.mean(lengths)

def path_length(x, T, h):
  if T.size is not None:
    return h + T.size
  q = T.split_dim
  if x[q - 1] < T.split_thresh:
    return path_length(x, T.left, h+1)
  else:
    return path_length(x, T.right, h+1)


In [9]:
import pandas as pd

columns = ["duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land", "wrong_fragment", "urgent",
        "hot", "num_failed_logins", "logged_in", "num_compromised", "root_shell", "su_attempted", "num_root",
        "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
        "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
        "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
        "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
        "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label"]

df = pd.read_csv("/content/kddcup.data.corrected", sep=",", names=columns, index_col=None)

In [12]:
df_numeric = df.select_dtypes(include=[np.number])
X = df_numeric.values

In [16]:
forest = isolation_forest(X, n=100, sample_size=256)

scores = [anomaly_score(forest, x) for x in X[:1000]]

In [18]:
df_sample = df.iloc[:1000]
df_sample = df_sample.copy()

scores = [anomaly_score(forest, x) for x in X[:1000]]

df_sample['score'] = scores
df_sample['anomaly'] = df_sample['score'] < np.percentile(scores, 10)

print(df_sample[['label', 'score', 'anomaly']].head(20))

      label  score  anomaly
0   normal.   1.54    False
1   normal.   1.54    False
2   normal.   1.53    False
3   normal.   1.53    False
4   normal.   1.53    False
5   normal.   1.53    False
6   normal.   1.53    False
7   normal.   1.53    False
8   normal.   1.53    False
9   normal.   1.53    False
10  normal.   1.53    False
11  normal.   1.53    False
12  normal.   1.53    False
13  normal.   1.53    False
14  normal.   1.53    False
15  normal.   1.53    False
16  normal.   1.53    False
17  normal.   1.53    False
18  normal.   1.53    False
19  normal.   1.53    False
